In [1]:
!pip install transformers datasets gradio torch scikit-learn seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import torch
import gradio as gr
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import classification_report

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
!pip install pytesseract pillow pdf2image -q
!apt-get install tesseract-ocr -q
!apt-get install poppler-utils -q

import pytesseract
from PIL import Image
from pdf2image import convert_from_path
from google.colab import files

def extract_text_from_image(image_path):
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image)
    return text

def extract_text_from_pdf(pdf_path):
    pages = convert_from_path(pdf_path)
    full_text = ""
    for page in pages:
        text = pytesseract.image_to_string(page)
        full_text += text + "\n"
    return full_text

print("OCR functions ready!")
print("You can now extract text from scanned contract images or PDFs!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (193 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.

In [5]:
from datasets import load_dataset

dataset = load_dataset("lex_glue", "ledgar", trust_remote_code=True)
print(dataset)
print(dataset["train"][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in a

README.md: 0.00B [00:00, ?B/s]

ledgar/train-00000-of-00001.parquet:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

ledgar/test-00000-of-00001.parquet:   0%|          | 0.00/3.31M [00:00<?, ?B/s]

ledgar/validation-00000-of-00001.parquet:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})
{'text': 'Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.', 'label': 97}


In [6]:
labels = dataset["train"].features["label"].names
print(f"Total clause types: {len(labels)}")
print("\nClause types:")
for i, label in enumerate(labels):
    print(f"{i}: {label}")

Total clause types: 100

Clause types:
0: Adjustments
1: Agreements
2: Amendments
3: Anti-Corruption Laws
4: Applicable Laws
5: Approvals
6: Arbitration
7: Assignments
8: Assigns
9: Authority
10: Authorizations
11: Base Salary
12: Benefits
13: Binding Effects
14: Books
15: Brokers
16: Capitalization
17: Change In Control
18: Closings
19: Compliance With Laws
20: Confidentiality
21: Consent To Jurisdiction
22: Consents
23: Construction
24: Cooperation
25: Costs
26: Counterparts
27: Death
28: Defined Terms
29: Definitions
30: Disability
31: Disclosures
32: Duties
33: Effective Dates
34: Effectiveness
35: Employment
36: Enforceability
37: Enforcements
38: Entire Agreements
39: Erisa
40: Existence
41: Expenses
42: Fees
43: Financial Statements
44: Forfeitures
45: Further Assurances
46: General
47: Governing Laws
48: Headings
49: Indemnifications
50: Indemnity
51: Insurances
52: Integration
53: Intellectual Property
54: Interests
55: Interpretations
56: Jurisdictions
57: Liens
58: Litigatio

In [7]:
model_checkpoint = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("Tokenizer loaded!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded!


In [8]:
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = dataset["train"].map(tokenize, batched=True)
tokenized_val = dataset["validation"].map(tokenize, batched=True)
tokenized_test = dataset["test"].map(tokenize, batched=True)

print(f"Train: {len(tokenized_train)}")
print(f"Validation: {len(tokenized_val)}")
print(f"Test: {len(tokenized_test)}")

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train: 60000
Validation: 10000
Test: 10000


In [9]:
small_train = tokenized_train.select(range(500))
small_val = tokenized_val.select(range(100))
small_test = tokenized_test.select(range(100))

print(f"Small Train: {len(small_train)}")
print(f"Small Val: {len(small_val)}")
print(f"Small Test: {len(small_test)}")

Small Train: 500
Small Val: 100
Small Test: 100


In [10]:
num_labels = len(dataset["train"].features["label"].names)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels
)
model.to(device)
print(f"Model loaded with {num_labels} output labels!")

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Model loaded with 100 output labels!


In [11]:
training_args = TrainingArguments(
    output_dir="./ledgar_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()
)
print("Training arguments set!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set!


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

print("Metrics function ready!")

Metrics function ready!


In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
print("Trainer ready!")

Trainer ready!


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,4.499146,4.442129,0.140000,0.074210
2,4.309753,4.290898,0.210000,0.118506
3,4.091492,4.226992,0.200000,0.091479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=96, training_loss=4.33444086710612, metrics={'train_runtime': 125.8154, 'train_samples_per_second': 11.922, 'train_steps_per_second': 0.763, 'total_flos': 395013851136000.0, 'train_loss': 4.33444086710612, 'epoch': 3.0})

In [15]:
results = trainer.evaluate(small_test)
print("\nTest Results:")
for key, value in results.items():
    print(f"{key}: {round(value, 4)}")



Test Results:
eval_loss: 4.3107
eval_accuracy: 0.17
eval_f1: 0.0785
eval_runtime: 0.8491
eval_samples_per_second: 117.775
eval_steps_per_second: 8.244
epoch: 3.0


In [16]:
model.save_pretrained("./ledgar_model_final")
tokenizer.save_pretrained("./ledgar_model_final")
print("Model saved!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [17]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

import spacy
import re

nlp = spacy.load("en_core_web_sm")

def extract_entities(contract_text):
    doc = nlp(contract_text)
    obligors = []
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG"]:
            obligors.append(ent.text)

    dates = []
    for ent in doc.ents:
        if ent.label_ == "DATE":
            dates.append(ent.text)

    date_patterns = [
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b',
        r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b',
        r'\bmaturity date[:\s]+([^\n,]+)',
        r'\bexpir(?:es|ation)[:\s]+([^\n,]+)'
    ]

    for pattern in date_patterns:
        matches = re.findall(pattern, contract_text, re.IGNORECASE)
        dates.extend(matches)
    obligors = list(set(obligors))
    dates = list(set(dates))

    return {
        "Obligor Names": obligors if obligors else ["Not found"],
        "Maturity Dates": dates if dates else ["Not found"]
    }

sample = """
This agreement is made between JPMorgan Chase & Co. and ABC Corporation.
The maturity date of this contract is December 31, 2025.
John Smith is the authorized signatory and obligor for this agreement.
The contract expires on 12/31/2025.
"""

result = extract_entities(sample)
print("Extracted Entities:")
print(f"Obligor Names: {result['Obligor Names']}")
print(f"Maturity Dates: {result['Maturity Dates']}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Extracted Entities:
Obligor Names: ['John Smith', 'ABC Corporation', 'JPMorgan Chase & Co.']
Maturity Dates: ['of this contract is December 31', '12/31/2025', 'December 31, 2025', 'on 12/31/2025.']


In [18]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./ledgar_model_final",
    tokenizer="./ledgar_model_final",
    device=0 if torch.cuda.is_available() else -1
)

labels = dataset["train"].features["label"].names

def classify_clause(text):
    result = classifier(text, truncation=True, max_length=512)
    label_index = int(result[0]["label"].split("_")[-1])
    clause_type = labels[label_index]
    confidence = round(result[0]["score"] * 100, 2)
    return f"Clause Type: {clause_type}\nConfidence: {confidence}%"

print("Inference function ready!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inference function ready!


In [19]:
import gradio as gr

custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    font-family: 'Segoe UI', sans-serif;
}
.main-header {
    text-align: center;
    padding: 30px;
    background: linear-gradient(90deg, #0f3460, #533483);
    border-radius: 15px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
}
.main-header h1 {
    color: #e94560 !important;
    font-size: 2.5em !important;
    font-weight: 800 !important;
    margin: 0 !important;
}
.main-header p {
    color: #a8b2d8 !important;
    font-size: 1.1em !important;
    margin-top: 10px !important;
}
.input-section {
    background: rgba(255,255,255,0.05);
    border-radius: 12px;
    padding: 20px;
    border: 1px solid rgba(233,69,96,0.3);
}
.output-section {
    background: rgba(255,255,255,0.05);
    border-radius: 12px;
    padding: 20px;
    border: 1px solid rgba(83,52,131,0.5);
}
.gr-button-primary {
    background: linear-gradient(90deg, #e94560, #533483) !important;
    border: none !important;
    border-radius: 8px !important;
    color: white !important;
    font-weight: 700 !important;
    font-size: 1.1em !important;
    padding: 12px 30px !important;
}
.gr-button-secondary {
    background: rgba(255,255,255,0.1) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    border-radius: 8px !important;
    color: white !important;
}
label {
    color: #a8b2d8 !important;
    font-weight: 600 !important;
}
textarea, input {
    background: rgba(255,255,255,0.07) !important;
    border: 1px solid rgba(233,69,96,0.3) !important;
    border-radius: 8px !important;
    color: white !important;
}
.gr-dropdown select {
    background: rgba(255,255,255,0.07) !important;
    color: white !important;
    border: 1px solid rgba(233,69,96,0.3) !important;
    border-radius: 8px !important;
}
footer {display: none !important;}
"""

def full_contract_analysis(contract_text):
    if not contract_text.strip():
        return "⚠️ Please paste contract text to analyze!"

    # Clause classification
    result = classifier(contract_text, truncation=True, max_length=512)
    label_index = int(result[0]["label"].split("_")[-1])
    clause_type = labels[label_index]
    confidence = round(result[0]["score"] * 100, 2)

    # NER extraction
    entities = extract_entities(contract_text)

    # Confidence bar
    conf_bar = "█" * int(confidence // 10) + "░" * (10 - int(confidence // 10))

    output = f"""
╔══════════════════════════════════════════╗
         ⚖️  CONTRACT ANALYSIS REPORT
╚══════════════════════════════════════════╝

📌 CLAUSE CLASSIFICATION
──────────────────────────────────────────
  Type       : {clause_type}
  Confidence : {conf_bar} {confidence}%

👤 OBLIGOR NAMES IDENTIFIED
──────────────────────────────────────────
{chr(10).join(f"  • {name}" for name in entities['Obligor Names'])}

📅 MATURITY DATES IDENTIFIED
──────────────────────────────────────────
{chr(10).join(f"  • {date}" for date in entities['Maturity Dates'])}

══════════════════════════════════════════
  ✅ Analysis Complete — Powered by Legal-BERT
══════════════════════════════════════════
"""
    return output


with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:

    # Header
    gr.HTML("""
    <div class='main-header'>
        <h1>⚖️ Contract Analysis System</h1>
        <p>Powered by Legal-BERT | Inspired by JPMorgan's NLP Pipeline</p>
        <p style='color:#e94560; font-size:0.9em;'>
            🔍 Clause Classification &nbsp;|&nbsp;
            👤 Obligor Detection &nbsp;|&nbsp;
            📅 Date Extraction
        </p>
    </div>
    """)

    with gr.Row():
        # Left column - Input
        with gr.Column(scale=1, elem_classes="input-section"):
            gr.HTML("<h3 style='color:#e94560;'>📄 Input Contract</h3>")
            contract_input = gr.Textbox(
                lines=18,
                placeholder="Paste your legal contract text here...\n\nExample:\nThis agreement is made between JPMorgan Chase & Co. and ABC Corporation...",
                label="Contract Text",
                show_label=True
            )
            with gr.Row():
                submit_btn = gr.Button("🔍 Analyze Contract", variant="primary")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary")

            # Sample button
            gr.HTML("<p style='color:#a8b2d8; font-size:0.85em; margin-top:10px;'>💡 Tip: Use the sample text below to test</p>")
            sample_btn = gr.Button("📋 Load Sample Contract", variant="secondary")

        # Right column - Output
        with gr.Column(scale=1, elem_classes="output-section"):
            gr.HTML("<h3 style='color:#533483;'>📊 Analysis Report</h3>")
            output_box = gr.Textbox(
                lines=18,
                label="Results",
                show_label=True,
                interactive=False
            )

    # Info cards at bottom
    gr.HTML("""
    <div style='display:flex; gap:15px; margin-top:20px;'>
        <div style='flex:1; background:rgba(233,69,96,0.1); border:1px solid rgba(233,69,96,0.3);
                    border-radius:10px; padding:15px; text-align:center;'>
            <div style='color:#e94560; font-size:1.5em;'>📌</div>
            <div style='color:white; font-weight:700;'>Clause Classification</div>
            <div style='color:#a8b2d8; font-size:0.85em;'>60+ clause types identified</div>
        </div>
        <div style='flex:1; background:rgba(83,52,131,0.1); border:1px solid rgba(83,52,131,0.5);
                    border-radius:10px; padding:15px; text-align:center;'>
            <div style='color:#533483; font-size:1.5em;'>👤</div>
            <div style='color:white; font-weight:700;'>Entity Recognition</div>
            <div style='color:#a8b2d8; font-size:0.85em;'>Obligor names extracted</div>
        </div>
        <div style='flex:1; background:rgba(15,52,96,0.3); border:1px solid rgba(15,52,96,0.5);
                    border-radius:10px; padding:15px; text-align:center;'>
            <div style='color:#4fc3f7; font-size:1.5em;'>📅</div>
            <div style='color:white; font-weight:700;'>Date Extraction</div>
            <div style='color:#a8b2d8; font-size:0.85em;'>Maturity dates detected</div>
        </div>
    </div>
    """)

    # Sample contract text
    sample_text = """This Loan Agreement is entered into as of January 15, 2024,
by and between JPMorgan Chase & Co., a corporation organized
under the laws of the State of New York, and ABC Corporation,
a Delaware corporation, and John Smith as the authorized
obligor and guarantor.

The maturity date of this loan shall be December 31, 2026.
All outstanding principal and interest shall be due and
payable on the maturity date.

Either party may terminate this agreement upon 30 days
written notice to the other party.

This Agreement shall be governed by the laws of the
State of New York.

Signed by:
John Smith - Obligor
Mary Johnson - Witness
JPMorgan Chase & Co. - Lender
ABC Corporation - Borrower"""

    # Button actions
    submit_btn.click(fn=full_contract_analysis, inputs=contract_input, outputs=output_box)
    clear_btn.click(fn=lambda: ("", ""), outputs=[contract_input, output_box])
    sample_btn.click(fn=lambda: sample_text, outputs=contract_input)

demo.launch(share=True)

/tmp/ipykernel_4798/1042318502.py:114: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_4798/1042318502.py:114: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://351190df68ac5bf91b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
